# 06 · Applying the trained models to the CRISP sites

Runs each trained model on live Sentinel imagery of its CRISP sites for real events, and writes maps and a
summary table. Imagery: Microsoft Planetary Computer (free, no account). **Run notebooks 01–04 first**; this one
loads their checkpoints from Drive.

| Module | Sites | Imagery | Output |
|---|---|---|---|
| Floods | 6 flood events (Sukkur 2022, Valencia 2024, …) | Sentinel-1 radar, before & after | new-water area (km²) |
| Wildfires | 5 fires (Pedrógão 2017, Tubbs 2017, …) | Sentinel-2, before & after | burned area (km²) |
| Forestry | 4 forest sites | Sentinel-2, two years apart | forest cover & loss (km²) |
| Agriculture | 6 farm sites + 36 Spain/UK sites | Sentinel-2, growing season | land-use composition |

These are **applications, not accuracy tests** — the test-set scores are in notebooks 01–04. Comparing the
mapped areas with published event reports is a useful sanity check for the write-up. Runtime: 20–40 min (mostly downloads).

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "crisp_sites"
from gstp import inference as INF, viz
from gstp.models import build_unet, build_s2_classifier
from gstp.data.raster import normalise
from gstp.utils import load_json, load_sites, save_json, CONFIG_DIR
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt, traceback

RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE
(RESULTS / "maps").mkdir(parents=True, exist_ok=True)
EVENTS = load_json(CONFIG_DIR / "crisp_events.json")
SITES = {s["id"]: s for s in load_sites()}
DEV = get_device()

def load_seg(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m = build_unet(ck["config"]["in_channels"], pretrained=False); m.load_state_dict(ck["state_dict"])
    return m.eval().to(DEV), ck["config"], ck.get("threshold", 0.5)

def save_map(panels, titles, title, path):
    fig, ax = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2))
    for a, p, t in zip(np.atleast_1d(ax), panels, titles):
        a.imshow(p); a.set_title(t, fontsize=9); a.axis("off")
    fig.suptitle(title, fontsize=10); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

## Floods — Sentinel-1 radar model, change between before and after

In [ ]:
flood_m, flood_cfg, flood_thr = load_seg(CKPT_DIR / "floods" / "flood_s1_best.pt")
st = flood_cfg["extra"]["stats"]
rows = []
for ev in EVENTS["floods"]:
    site = SITES[ev["site"]]
    try:
        out = {}
        for when in ("pre", "post"):
            arr, valid, info = INF.load_s1_db(site["bbox"], ev[when])
            prob = INF.sliding_window_probs(flood_m, normalise(arr, "zscore", st), device=DEV)
            out[when] = (arr, (prob >= flood_thr) & valid, info)
        new_water = out["post"][1] & ~out["pre"][1]
        rows.append({"site": site["label"], "event": ev["event"], "pre_date": out["pre"][2]["date"],
                     "post_date": out["post"][2]["date"], "water_before_km2": INF.area_km2(out["pre"][1], 10),
                     "water_after_km2": INF.area_km2(out["post"][1], 10), "new_water_km2": INF.area_km2(new_water, 10)})
        vv = lambda a: np.clip((a[0] + 25) / 25, 0, 1)
        save_map([vv(out["pre"][0]), vv(out["post"][0]), viz.overlay(np.dstack([vv(out["post"][0])] * 3), new_water, (0.1, 0.4, 1))],
                 [f"VV before {out['pre'][2]['date']}", f"VV after {out['post'][2]['date']}", "new flood water (blue)"],
                 f"{site['label']} — {ev['event']}", RESULTS / "maps" / f"flood_{site['id']}.png")
        print("✓", rows[-1])
    except Exception as e:
        print("✗", site["id"], e)
pd.DataFrame(rows).to_csv(RESULTS / "floods.csv", index=False)
pd.DataFrame(rows)

## Wildfires — burn-scar model on Sentinel-2 (30 m), burned after but not before

In [ ]:
burn_m, burn_cfg, burn_thr = load_seg(CKPT_DIR / "wildfires" / "burn_scar_best.pt")
ex = burn_cfg["extra"]; S2B = ["B02", "B03", "B04", "B8A", "B11", "B12"]
rows = []
for ev in EVENTS["wildfires"]:
    site = SITES[ev["site"]]
    try:
        out = {}
        for when in ("pre", "post"):
            arr, valid, info = INF.load_s2(site["bbox"], ev[when], S2B, resolution=30)
            x = arr / 10000.0 * ex["reflectance_scale"]
            prob = INF.sliding_window_probs(burn_m, normalise(x, "zscore", ex["stats"]), device=DEV)
            out[when] = (arr, (prob >= burn_thr) & valid, info)
        burned = out["post"][1] & ~out["pre"][1]
        rows.append({"site": site["label"], "event": ev["event"], "post_scenes": ", ".join(out["post"][2]["dates"]),
                     "burned_area_km2": INF.area_km2(burned, 30)})
        save_map([viz.rgb(out["pre"][0], (5, 3, 2)), viz.rgb(out["post"][0], (5, 3, 2)),
                  viz.overlay(viz.rgb(out["post"][0], (2, 1, 0)), burned, (1, 0.3, 0))],
                 ["before (SWIR/NIR/Red)", "after (SWIR/NIR/Red)", "new burn scar (orange)"],
                 f"{site['label']} — {ev['event']}", RESULTS / "maps" / f"burn_{site['id']}.png")
        print("✓", rows[-1])
    except Exception as e:
        print("✗", site["id"], e)
pd.DataFrame(rows).to_csv(RESULTS / "wildfires.csv", index=False)
pd.DataFrame(rows)

## Wildfires (flames) — active-fire model on every Sentinel-2 image taken during each fire

In [ ]:
af_m, af_cfg, af_thr = load_seg(CKPT_DIR / "wildfires_active_fire" / "active_fire_best.pt")
afx = af_cfg["extra"]; rows = []
for ev in EVENTS["active_fire"]:
    site = SITES[ev["site"]]
    try:
        scenes = INF.load_s2_scenes(site["bbox"], ev["during"], afx["sentinel2_bands"], resolution=30)
        best = None
        for date, refl, valid, sid in scenes:
            prob = INF.sliding_window_probs(af_m, normalise(refl, "zscore", afx["stats"]), tile=256, overlap=32, device=DEV)
            fire = (prob >= af_thr) & valid
            rows.append({"site": site["label"], "event": ev["event"], "date": date, "scene": sid,
                         "fire_pixels": int(fire.sum()), "fire_area_km2": INF.area_km2(fire, 30), "valid_fraction": float(valid.mean())})
            if best is None or fire.sum() > best[2].sum():
                best = (date, refl, fire)
        if best is not None:
            fc = np.dstack([best[1][0], best[1][1], best[1][2]]); fc = np.clip(fc / np.percentile(fc, 99), 0, 1)
            save_map([fc, viz.overlay(fc, best[2], (1.0, 0.9, 0.0), alpha=0.8)], [f"SWIR2/SWIR1/Blue {best[0]}", "detected flames (yellow)"],
                     f"{site['label']} — {ev['event']}", RESULTS / "maps" / f"activefire_{site['id']}.png")
        print("✓", site["id"], len(scenes), "scenes")
    except Exception as e:
        print("✗", site["id"], e)
pd.DataFrame(rows).to_csv(RESULTS / "active_fire.csv", index=False)
pd.DataFrame(rows).sort_values("fire_pixels", ascending=False).head(15)

## Forestry — forest cover in two years and the loss between them

In [ ]:
forest_m, _, forest_thr = load_seg(CKPT_DIR / "forestry" / "forest_best.pt")
rows = []
for ev in EVENTS["forestry"]:
    site = SITES[ev["site"]]
    try:
        out = {}
        for when in ("before", "after"):
            arr, valid, info = INF.load_s2(site["bbox"], ev[when], ["B04", "B03", "B02", "B08"], max_cloud=50)
            prob = INF.sliding_window_probs(forest_m, normalise(arr, "percentile"), device=DEV)
            out[when] = (arr, (prob >= forest_thr) & valid, valid)
        both = out["before"][2] & out["after"][2]
        loss = out["before"][1] & ~out["after"][1] & both
        rows.append({"site": site["label"], "period": ev["event"],
                     "forest_before_km2": INF.area_km2(out["before"][1] & both, 10),
                     "forest_after_km2": INF.area_km2(out["after"][1] & both, 10),
                     "forest_loss_km2": INF.area_km2(loss, 10), "valid_fraction": float(both.mean())})
        save_map([viz.rgb(out["before"][0]), viz.rgb(out["after"][0]), viz.overlay(viz.rgb(out["after"][0]), loss, (1, 0, 0))],
                 ["before", "after", "forest loss (red)"], f"{site['label']} — {ev['event']}",
                 RESULTS / "maps" / f"forest_{site['id']}.png")
        print("✓", rows[-1])
    except Exception as e:
        print("✗", site["id"], e)
pd.DataFrame(rows).to_csv(RESULTS / "forestry.csv", index=False)
pd.DataFrame(rows)

## Agriculture — land-use composition of every farm, Spain and UK site

In [ ]:
# Each site runs in its own process with a memory cap and is loaded in tiles (tools/apply_agriculture.py): a heavy Sentinel-2 load
# then fails that one site instead of crashing the Colab session, and results are saved after every site.
import json, os
AG_CKPT = CKPT_DIR / "agriculture" / "landuse_eurosat_best.pt"
targets = [s for s in SITES.values() if s["focus"] in ("agriculture", "all")]
part = RESULTS / "agriculture_sites"; part.mkdir(exist_ok=True)
env = {**os.environ, "CUDA_VISIBLE_DEVICES": ""}
for site in targets:
    out = part / f"{site['id']}.json"
    if out.exists():
        continue
    for attempt, (tiles, items) in enumerate([(2, 3), (3, 2)]):
        r = subprocess.run([sys.executable, "tools/apply_agriculture.py", str(AG_CKPT), site["id"], str(out),
                            "--mem-gb", "6", "--max-items", str(items), "--tiles", str(tiles)], capture_output=True, text=True, env=env)
        if out.exists():
            print("✓", site["id"]); break
        print("✗", site["id"], f"attempt {attempt + 1}:", (r.stderr.strip().splitlines() or ["?"])[-1][:140])
rows = [json.loads(p.read_text()) for p in sorted(part.glob("*.json"))]
agri = pd.DataFrame(rows)
agri["farmland_share"] = agri[["AnnualCrop", "PermanentCrop", "Pasture"]].sum(1)
agri.sort_values("farmland_share", ascending=False).to_csv(RESULTS / "agriculture_landuse.csv", index=False)
agri.sort_values("farmland_share", ascending=False).head(15)

In [ ]:
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip:", archive)